In [2]:
from google.colab import files
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()

# Browse and select multiple CSV files
uploaded = files.upload()

# Load selected CSV files into PySpark DataFrames
dataframes = {}

for filename in uploaded.keys():
    df = spark.read.csv(filename, header=True, inferSchema=True)
    dataframes[filename] = df

    print(f"\nFile: {filename}")
    print("Records:", df.count())
    df.show(5)

Saving lazy_execution_products (1).csv to lazy_execution_products (1).csv
Saving lazy_execution_sales_25000 (1).csv to lazy_execution_sales_25000 (1).csv
Saving lazy_execution_stores (1).csv to lazy_execution_stores (1).csv

File: lazy_execution_products (1).csv
Records: 15
+----------+-------------------+-----------+----------+----------+
|product_id|       product_name|   category|list_price|cost_price|
+----------+-------------------+-----------+----------+----------+
|      P001|     Wireless Mouse|Accessories|       799|       420|
|      P002|Mechanical Keyboard|Accessories|      2999|      1600|
|      P003|          USB-C Hub|Accessories|      1899|       920|
|      P004|    27-inch Monitor|   Displays|     16999|     11400|
|      P005|          Webcam HD|Accessories|      2499|      1400|
+----------+-------------------+-----------+----------+----------+
only showing top 5 rows

File: lazy_execution_sales_25000 (1).csv
Records: 25000
+--------+----------+--------+----------+

In [3]:
from pyspark.sql import functions as F
from pyspark import StorageLevel
import time

# Find the sales dataset from the uploaded files
sales_df = None

for filename, df in dataframes.items():
    if {"order_id", "quantity", "unit_price", "order_status"}.issubset(df.columns):
        sales_df = df
        print("Sales file selected:", filename)
        break

if sales_df is None:
    raise ValueError("Sales CSV not found. Please upload the sales dataset.")

print("Total Sales Records:", sales_df.count())
sales_df.show(5)

Sales file selected: lazy_execution_sales_25000 (1).csv
Total Sales Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Comple

In [4]:
#Transformation
completed_df=sales_df.filter(F.col("order_status")=="Completed")
print("Transformation created")
#ACTION 1
print("Completed Orders: ",completed_df.count())
#ACTION 2
completed_df.show(10)

Transformation created
Completed Orders:  21775
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
| O000006|2026-0

In [34]:
#Cache the filtered DataFrame
completed_df.cache()
print("Is cached: ",completed_df.is_cached)
#First action populates cache
completed_df.count()
#Reuse cached data
completed_df.show(5)
#Another action using cached data
completed_df.groupBy("product_id").count().show(5)

Is cached:  True
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Completed|       Online|
+--------+----------+--------+----------+------

PERSIST- STORING
UNPERSIST - REMOVING FROM MEMORY

In [6]:
#Create filtered DataFrame
high_quantity_df=sales_df.filter(F.col("quantity")>5)
#Persist using Memory and DISK
high_quantity_df.persist(StorageLevel.MEMORY_AND_DISK)
#Trigger Execution
high_quantity_df.count()
#Reuse
high_quantity_df.show(10)
#check storage level
print(high_quantity_df.storageLevel)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8|      2499|           0|           UPI|   Completed|     In Store|
| O000007|2026-02-27|     S03|      P010|       8|      3999|          10|           UPI|   Completed|       Online|
| O000008|2026-03-21|     S07|      P002|       9|      2999|   

In [7]:
#Check cache status
print("Before: ",completed_df.is_cached)
#Remove cache
completed_df.unpersist(blocking=True)
#Check again
print("After: ",completed_df.is_cached)
#REmove other persisted data frame
high_quantity_df.unpersist(blocking=True)


Before:  True
After:  False


DataFrame[order_id: string, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: int, discount_pct: int, payment_method: string, order_status: string, sales_channel: string]

In [8]:
# Generate 40 batch IDs (0 to 39)
batch_df = spark.range(40).select(
    F.col("id").alias("batch_id")
)

batch_df.show(5)

# Combine each sales record with all 40 batch IDs
large_df = sales_df.crossJoin(batch_df)

print("Original Records:", sales_df.count())
print("Large Dataset Records:", large_df.count())

large_df.show(5)

+--------+
|batch_id|
+--------+
|       0|
|       1|
|       2|
|       3|
|       4|
+--------+
only showing top 5 rows
Original Records: 25000
Large Dataset Records: 1000000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|batch_id|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       0|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       1|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       2|
| O000001|2026-06-22|     S04|      P002|       3|      2999|     

In [9]:
# Create transformations
result_df = large_df.filter(
    F.col("quantity") > 3
).withColumn(
    "revenue",
    F.col("quantity") * F.col("unit_price")
).groupBy("product_id").agg(
    F.sum("revenue").alias("total_revenue")
)

# First execution
start = time.perf_counter()

result_df.collect()

time1 = time.perf_counter() - start

# Second execution
start = time.perf_counter()

result_df.collect()

time2 = time.perf_counter() - start

print("First Execution:", round(time1, 3), "seconds")
print("Second Execution:", round(time2, 3), "seconds")

First Execution: 2.457 seconds
Second Execution: 0.177 seconds


In [10]:
#Check total records
print("Total Records: ",sales_df.count())
#check number of partitions
print("Numbr of Partitions: ",sales_df.rdd.getNumPartitions())


Total Records:  25000
Numbr of Partitions:  1


In [12]:
#Create 8 partitions
partitioned_df=sales_df.repartition(8)
print("Original Partitions: ",sales_df.rdd.getNumPartitions())
print("New Partitions: ",partitioned_df.rdd.getNumPartitions())
#Trigger execution
partitioned_df.count()


Original Partitions:  1
New Partitions:  8


25000

In [13]:
from pyspark.sql import functions as F

# Add partition ID to every record
distribution_df = partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)

# Count records in each partition
distribution_df.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 3125|
|           1| 3125|
|           2| 3125|
|           3| 3125|
|           4| 3125|
|           5| 3125|
|           6| 3125|
|           7| 3125|
+------------+-----+



KEY BASED PARTITIONING


In [14]:
#Partition data by store_id
stored_partitioned_df=sales_df.repartition(8,"store_id")
print("Partitions: ",stored_partitioned_df.rdd.getNumPartitions())

#Inspect partition distribution
stored_partitioned_df.withColumn("partition_id",F.spark_partition_id()
).groupBy("partition_id","store_id").count().orderBy("partition_id","store_id").show(30)

Partitions:  8
+------------+--------+-----+
|partition_id|store_id|count|
+------------+--------+-----+
|           0|     S01| 3180|
|           1|     S02| 3121|
|           1|     S08| 3122|
|           3|     S06| 3033|
|           4|     S05| 3117|
|           4|     S07| 3209|
|           5|     S03| 3132|
|           5|     S04| 3086|
+------------+--------+-----+



eg: Say you have 6 boxes and want 3.

Repartition(3): Dump all the data from the 6 boxes, shuffle it all together, and fill 3 brand-new boxes evenly.

Coalesce(3): Keep 3 of the boxes as they are and pour the contents of the other 3 into them. No full shuffle happens, so the data mostly stays where it already is.

In [15]:
#create 8 partitions
df8=sales_df.repartition(8)
#Reduce using repartition
df_repartition=df8.repartition(3)
#Reduce using coalesce
df_coalesce=df8.coalesce(3)
print("Repartition: ",df_repartition.rdd.getNumPartitions())
print("Coalesce: ",df_coalesce.rdd.getNumPartitions())
#Examine execution plans
print("REPARTITION PLAN")
df_repartition.explain()
print("COALESCE PLAN")
df_coalesce.explain()

Repartition:  3
Coalesce:  3
REPARTITION PLAN
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1145]
         +- FileScan csv [order_id#99,order_date#100,store_id#101,product_id#102,quantity#103,unit_price#104,discount_pct#105,payment_method#106,order_status#107,sales_channel#108] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000 (1).csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1145]
   +- FileScan csv [order_id#99,order_date#100,store_id#101,product_id#102,quantity#103,unit_price#104,discount_pct#105,payment_method#106,order_status#107,sales_channel#108] Batched: fa

TASKS TO DO:
```
1. Find the current number of partitions.
2. Increase the number of partitions to 12.
3. Display the number of records in each partition.
4. Repartition the DataFrame into 6 partitions using product_id.
5. Reduce 12 partitions to 4 using coalesce().
```

In [21]:
sales_df.rdd.getNumPartitions()

1

In [24]:
df_1=sales_df.repartition(12)
df_1.rdd.getNumPartitions()

12

In [25]:
df_1.withColumn("partition_id", F.spark_partition_id()
).groupBy("partition_id").count().orderBy("partition_id").show(30)

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 2084|
|           1| 2083|
|           2| 2083|
|           3| 2083|
|           4| 2083|
|           5| 2083|
|           6| 2083|
|           7| 2083|
|           8| 2083|
|           9| 2084|
|          10| 2084|
|          11| 2084|
+------------+-----+



In [31]:
df_repartition=df_1.repartition(6,"product_id")
df_repartition.show(5)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000008|2026-03-21|     S07|      P002|       9|      2999|           0|          Card|   Completed|     In Store|
| O000017|2026-03-19|     S05|      P004|       9|     16999|           0|          Cash|   Completed|       Online|
| O000018|2026-05-15|     S07|      P012|       7|      6999|           5|          Cash|   Completed|       Online|
+--------+----------+--------+----------+--------+----------+---

In [32]:
df_coalesce=df_1.coalesce(3)
df_coalesce.show(5)

+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000008|2026-03-21|     S07|      P002|       9|      2999|           0|          Card|   Completed|     In Store|
| O000017|2026-03-19|     S05|      P004|       9|     16999|           0|          Cash|   Completed|       Online|
| O000018|2026-05-15|     S07|      P012|       7|      6999|           5|          Cash|   Completed|       Online|
+--------+----------+--------+----------+--------+----------+---